# Extra · grid de hiperparámetros

> **No es el eje del módulo.** El ebook ya no pasa por aquí. Queda si quieres oír, con el juguete, por qué “el LR correcto” no se adivina.

## Dummy

Un hiperparámetro no se aprende con backprop. Lo eliges tú: learning rate, batch, profundidad, ancho. El **grid** es el experimento más tonto y más honesto: pruebas combinaciones, anotas val_loss, te quedas con las que no explotaron.

No busques “el óptimo universal”. Busca *una región estable*. En el gym de LoRA vas a elegir LR $2\times 10^{-4}$ y $r=16$ con esta misma cabeza: no porque un paper lo grabó en piedra, porque en *tu* curva no se fue a `NaN`.

## Matemática

Quieres

$$
\lambda^\star \in \arg\min_{\lambda\in\Lambda}\;\mathcal{L}_{\mathrm{val}}\bigl(\theta^\star(\lambda)\bigr),
$$

donde $\theta^\star(\lambda)$ es lo que Adam encontró *entrenando* con esos hparams $\lambda$ (lr, $B$, $n_{\mathrm{layer}}$, $d_{\mathrm{model}}$). El grid es enumerar un $\Lambda$ finito. Es costoso: $|\Lambda|$ entrenamientos.

Regla práctica: el LR es el dial más violento. $10^{-3}$ a veces explota; $10^{-4}$ a veces ni se mueve. Batch grande suaviza el gradiente (menos ruido, más VRAM). Más capas / más $d$ = más capacidad = más overfit si no hay datos.

$$
\hat g = \frac{1}{B}\sum_{i=1}^{B}\nabla_\theta \ell(x^{(i)};\theta)
$$

es un promedio. $B$ chico → $\hat g$ nervioso. $B$ grande → $\hat g$ calmo y caro.

## Cómo leer la tabla

`stable=False` o `val_loss=inf` no es un bug de tu lápiz: esa esquina del grid **no se puede habitar**. El top-10 por val_loss es tu mapa. En CPU recortamos el grid; si lo dejas completo (3 lr × 3 batch × 2 layers × 2 anchos × 3 épocas) te come la tarde.


In [ ]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEXT_PATH = "quijote.txt"
print(f"device={DEVICE}  archivo={TEXT_PATH}")

import time
from typing import Dict, Any


In [ ]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [ ]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 2, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.seq_len, "T excede seq_len del modelo"
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total += loss.item()
        n += 1
    return total / max(1, n)


@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, seq_len: int, max_new_tokens: int, temperature: float, device: str):
    model.eval()
    device = torch.device(device)
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out = ids[:]
    for _ in range(max_new_tokens):
        ctx = out[-seq_len:]
        x = torch.tensor(ctx, dtype=torch.long, device=device).unsqueeze(0)
        logits = model(x)[0, -1, :] / max(1e-8, temperature)
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1).item()
        out.append(next_id)
    return decode(out, itos)


Grid chico para CPU. Descomenta las tuplas largas si hay GPU y tiempo. Ordenamos por `val_loss` al final: abajo salen las peores (incluidas las que se fueron a `inf`).


In [ ]:
@dataclass
class GridConfig:
    seq_len: int = 64
    n_head: int = 4
    dropout: float = 0.1
    epochs: int = 2
    max_windows: int = 3000
    device: str = DEVICE
    lrs: Tuple[float, ...] = (1e-4, 3e-4)
    batch_sizes: Tuple[int, ...] = (16, 32)
    n_layers: Tuple[int, ...] = (2,)
    n_embds: Tuple[int, ...] = (128,)


def run_grid(text_path: str, cfg: GridConfig):
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    ids = encode(text, stoi)
    n = len(ids)
    split = int(0.9 * n)
    train_ds = CharWindowDataset(ids[:split], cfg.seq_len, cfg.max_windows, seed=0)
    val_ds = CharWindowDataset(ids[split:], cfg.seq_len, min(800, cfg.max_windows), seed=1)
    device = torch.device(cfg.device)
    vocab = len(stoi)
    results: List[Dict[str, Any]] = []

    for lr in cfg.lrs:
        for bs in cfg.batch_sizes:
            for nl in cfg.n_layers:
                for ne in cfg.n_embds:
                    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True)
                    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, drop_last=True)
                    model = MiniTransformerLM(vocab, cfg.seq_len, ne, cfg.n_head, nl, cfg.dropout).to(device)
                    optim = torch.optim.AdamW(model.parameters(), lr=lr)
                    stable = True
                    t0 = time.time()
                    try:
                        for _ in range(cfg.epochs):
                            tr = train_one_epoch(model, train_loader, optim, device)
                            if not math.isfinite(tr):
                                stable = False
                                break
                        vl = eval_loss(model, val_loader, device) if stable else float("inf")
                    except Exception:
                        stable = False
                        vl = float("inf")
                    dt = time.time() - t0
                    ppl = math.exp(vl) if (stable and vl < 20) else float("inf")
                    rec = {"lr": lr, "batch": bs, "layers": nl, "embd": ne, "val_loss": vl, "val_ppl": ppl, "time_s": round(dt, 2), "stable": stable}
                    results.append(rec)
                    print(f"lr={lr:.0e} bs={bs:2d} L={nl} d={ne:3d} | stable={stable} | val_loss={vl:.4f} | t={dt:.1f}s")
    return sorted(results, key=lambda r: r["val_loss"])


cfg = GridConfig()
results = run_grid(TEXT_PATH, cfg)
print("\nTOP 10")
for r in results[:10]:
    print(r)
print("\nPeores")
for r in results[-5:]:
    print(r)
